# Sentiment Analysis with Masks

In [1]:
import tensorflow as tf
from typing import Tuple

2023-04-05 09:43:56.603778: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Dataset

In [2]:
import tensorflow_datasets as tfds

datasets, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True)

2023-04-05 09:43:58.239537: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-05 09:43:58.264438: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-05 09:43:58.264625: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-05 09:43:58.265007: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropri

## Preprocessing

In [3]:
def preprocess(x_batch: tf.data.Dataset, y_batch: tf.data.Dataset) -> Tuple[tf.data.Dataset, tf.data.Dataset]:
    x_batch = tf.strings.substr(x_batch, 0, 300)
    x_batch = tf.strings.regex_replace(x_batch, b"<br\\s*/?>", b" ")
    x_batch = tf.strings.regex_replace(x_batch, b"[^a-zA-Z]", b" ")
    x_batch = tf.strings.split(x_batch)
    return x_batch.to_tensor(default_value=b"<pad>"), y_batch

In [4]:
from collections import Counter

vocab = Counter()
for x_batch, y_batch in datasets["train"].batch(32).map(preprocess):
    for review in x_batch:
        vocab.update(list(review.numpy()))

In [5]:
size_vocab = 10_000
vocab_trunc = [word for word, count in vocab.most_common()[:size_vocab]]
words_tensor = tf.constant(vocab_trunc)
ids_words = tf.range(len(vocab_trunc), dtype=tf.int64)
vocab_init = tf.lookup.KeyValueTensorInitializer(words_tensor, ids_words)
num_oov_buckets = 1_000
table = tf.lookup.StaticVocabularyTable(vocab_init, num_oov_buckets)
table

In [6]:
def encode_words(x_batch: tf.data.Dataset, y_batch: tf.data.Dataset) -> Tuple[tf.data.Dataset, tf.data.Dataset]:
    return table.lookup(x_batch), y_batch

train_set = datasets["train"].batch(32).map(preprocess).map(encode_words).prefetch(1)

## Model

In [7]:
size_embed = 128
inputs = tf.keras.layers.Input(shape=[None])
mask = tf.keras.layers.Lambda(lambda inp: tf.keras.backend.not_equal(inp, 0))(inputs)
z = tf.keras.layers.Embedding(input_dim=size_vocab+num_oov_buckets, output_dim=size_embed)(inputs)
z = tf.keras.layers.GRU(units=size_embed, return_sequences=True)(z, mask=mask)
z = tf.keras.layers.GRU(units=size_embed)(z, mask=mask)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(z)
model = tf.keras.Model(inputs=[inputs], outputs=[outputs])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, None)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, None, 128)    1408000     ['input_1[0][0]']                
                                                                                                  
 lambda (Lambda)                (None, None)         0           ['input_1[0][0]']                
                                                                                                  
 gru (GRU)                      (None, None, 128)    99072       ['embedding[0][0]',              
                                                                  'lambda[0][0]']             

In [8]:
import time
from pathlib import Path

root_logdir = Path().absolute() / "logs"

%load_ext tensorboard
%tensorboard --logdir=./logs --port=6006

Launching TensorBoard...

In [9]:
log_dir = root_logdir / time.strftime("run_%Y_%m_%d-%H_%M_%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir, histogram_freq=1)

In [10]:
history = model.fit(train_set, epochs=5, callbacks=[tensorboard_callback])

Epoch 1/5


2023-04-05 09:44:13.684881: W tensorflow/core/common_runtime/forward_type_inference.cc:332] Type inference failed. This indicates an invalid graph that escaped type checking. Error message: INVALID_ARGUMENT: expected compatible input types, but input 1:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_INT32
    }
  }
}
 is neither a subtype nor a supertype of the combined inputs preceding it:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_INT8
    }
  }
}

	while inferring type of node 'cond_41/output/_22'
2023-04-05 09:44:15.193177: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8401


782/782 [==============================] - 18s 14ms/step - loss: 0.5423 - accuracy: 0.7196
Epoch 2/5
782/782 [==============================] - 12s 15ms/step - loss: 0.3645 - accuracy: 0.8468
Epoch 3/5
782/782 [==============================] - 12s 15ms/step - loss: 0.2101 - accuracy: 0.9225
Epoch 4/5
782/782 [==============================] - 13s 16ms/step - loss: 0.1464 - accuracy: 0.9466
Epoch 5/5
782/782 [==============================] - 13s 17ms/step - loss: 0.1160 - accuracy: 0.9576


In [11]:
test_set = datasets["test"].batch(32).map(preprocess).map(encode_words).prefetch(1)
model.evaluate(test_set)

782/782 [==============================] - 8s 7ms/step - loss: 0.7968 - accuracy: 0.7245


[0.7967800498008728, 0.7245200276374817]